# Confidence Tiers — A Walkthrough

Every parameter in nidus carries one of four confidence tiers
describing the strength of evidence behind it. This notebook
explains what each tier means, how to read the
``tier_rationale`` field, and how tiers propagate through
derived quantities.

The tier system is the most rare and load-bearing thing about
nidus. Treat it as central, not decorative.

## Setup

In [ ]:
import nidus
from collections import Counter

ds = nidus.load()

## The tier definitions

In [ ]:
for tier_id, tier in ds.tiers.items():
    print(f"Tier {tier_id} — {tier.label}")
    for c in tier.criteria:
        print(f"  · {c}")
    print()

## Tier distribution in this dataset

The mix tells you something about the field: pregnancy
physiology is data-rich for maternal hemodynamics (mostly
Tier B), patchy for placental transport (a mix of B and C),
and almost entirely empty at Tier D (we have not yet started
cataloguing structured open questions).

In [ ]:
tier_counts = Counter(p.tier for p in ds)
for t in "ABCD":
    print(f"  Tier {t}: {tier_counts.get(t, 0):>3} parameters")

## How to read ``tier_rationale``

Every parameter carries a free-text rationale explaining why it
was assigned its tier. The rationale should reference the
evidence base — number of studies, cohort size, measurement
technique, known caveats.

Most rationales in v0.3 are placeholders synthesised from the
v0.2 TOML curation. Reviewers re-write them when promoting
parameters from ``unverified`` to ``verified``.

In [ ]:
example_b = next(p for p in ds if p.tier == "B")
print(f"[Tier B example] {example_b.id}")
print(example_b.tier_rationale)

In [ ]:
example_c = next(p for p in ds if p.tier == "C")
print(f"\n[Tier C example] {example_c.id}")
print(example_c.tier_rationale)

## Tier propagation rules

When a model derives a quantity from multiple parameters, the
derived quantity's tier is the **minimum** (lowest) of its
inputs. The reasoning is conservative:

* A derived quantity cannot be better-supported than its
  weakest input.
* A user comparing two derived quantities needs to know the
  worst-case provenance, not the best-case.

Concretely, if you compute ``F(p1, p2, p3)`` where p1 is
Tier A, p2 is Tier B, and p3 is Tier C, then the result is
Tier C.

Two additional degradations:

* Tier degrades by one level if a parameter is **extrapolated
  outside its validated range** (e.g. a 8–40 week parameter
  used at week 6).
* Tier degrades by one level if a parameter is **applied
  outside its applicability population** (e.g. a singleton
  parameter used for twins).

The next notebook demonstrates propagation in code.

## How to assign a tier

Tier inflation is a worse error than tier deflation. When in
doubt, choose the more conservative tier.

For a candidate parameter:

1. Is there a single citation, or multiple? Single → C or B.
2. Is the cohort longitudinal with n≥100? If yes → B candidate.
3. Are there ≥3 independent confirmations with overlapping
   confidence intervals? If yes → A candidate.
4. Is there no quantitative literature at all? → D.

See [CONTRIBUTING.md](../CONTRIBUTING.md) for the verification
standard, and the
[parameter-request issue template](https://github.com/clay-good/nidus/issues/new?template=parameter-request.yml)
for the full proposal flow.